# Lab 01: Vertical Profiles, Humidity Coordinates & Thermodynamics

Welcome to **Lab 01** of the **Tempest-ARKNET** Interactive Documentation Suite.

In this hands-on lab, you will run the exact physics and coordinate-encoding routines from `tempest/profiles/` and `tempest/generator/sample.py` (with pure-NumPy fallbacks from `tempest_weather/_core.py`) on a fast synthetic sounding—no large ERA5 `.zarr` archives or private model weights required.

**How to use this lab:** Before each main code cell, pause for 15 seconds on the **Predict → Run → Observe** callout and commit to a prediction before clicking **Run**. Active prediction makes the physical and numerical behavior stick far better than passive scrolling.

**What you'll explore (under 2 minutes on free Colab CPU):**
1. **The 44-Height AGL Grid & Vertical-Difference Encoding** (`to_diff` / `from_diff`)
2. **Log Dewpoint-Depression Humidity Coordinates** (`dpd_from_rh` / `rh_from_dpd`)
3. **2-Pass Hypsometric Atmospheric Column** (`thermo.column` → $p, \rho, a$)
4. **Interactive Site Elevation Shift** (`to_site` / `site_shift`) with live Colab sliders

### Before Running Setup: Why This Cell Checks the Python Version
Full launch-day orchestration (`tempest_weather_v2.launch`) uses Python 3.12+ f-string syntax, while Google Colab often defaults to Python 3.10 or 3.11. Running this setup cell adds the repository root to `sys.path` and lets us import the core physics submodules (`tempest.profiles.*` and `tempest_weather._core`) directly without any version friction.

In [ ]:
# @title 0. Environment Setup (Run Once)
MOUNT_DRIVE = False # @param {type:"boolean"}
REPO_URL = "https://github.com/senai-sorsa/Tempest-ARKNET.git" # @param {type:"string"}

import os, sys, subprocess, pathlib

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped (not running in Colab): {exc}")

repo_dir = pathlib.Path("/content/Tempest-ARKNET")
if not repo_dir.exists():
    # Allow running inside a local checkout if already inside the repo
    cwd = pathlib.Path.cwd()
    if (cwd / "tempest").is_dir() and (cwd / "tempest_weather_v2").is_dir():
        repo_dir = cwd
    else:
        print(f"Cloning {REPO_URL} into {repo_dir} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# Install lightweight dependencies quietly (torch is pre-installed on Google Colab)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['numpy', 'scipy', 'matplotlib']], check=False)

print(f"Python: {sys.version.split()[0]} | Repo ready at: {repo_dir}")
if sys.version_info < (3, 12):
    print("Note: Full tempest_weather_v2.launch requires Python >= 3.12 (PEP 701 f-strings at launch.py:684).")
    print("      This lab imports Python 3.10/3.11-compatible submodules directly and runs cleanly on Colab CPU.")

## 1. The 44-Height AGL Grid & Vertical-Difference Encoding

### Conceptual Intuition (Before the Math)
Imagine sketching the wind speed from a launch pad up to 18 km altitude. If you ask a neural network to predict wind at 10 m, 50 m, 100 m, …, 18,000 m as **44 separate, independent numbers**, tiny uncorrelated errors at neighboring altitudes turn into jagged, unphysical zig-zags—creating phantom wind-shear spikes that would over-stress a simulated rocket airframe.

Instead, Tempest-ARKNET uses an **elevator-step representation** (`tempest/profiles/diff.py::to_diff`):
- **Why keep index `0` (10 m AGL) as a raw value?** You need a single ground-floor anchor so the entire column has a known starting point.
- **Why encode indices `1..43` as layer differences (`value[k] - value[k-1]`)?** Neighboring altitudes in a real atmosphere are strongly correlated; their step differences stay tightly centered near zero, and the network learns *wind shear* and *temperature lapse rate* directly rather than as noisy subtractions of two large numbers.
- **Why is the 44-height grid non-uniform?** Twelve dense boundary-layer levels (`10, 50, 100, 150, 200, 300, 500, 750, 1000, 1250, 1500, 2000` m AGL) resolve rail-departure winds, surface inversions, and low-level shear where a rocket's dynamic pressure and angle-of-attack sensitivity are highest. Above `2000` m, the atmosphere varies more smoothly, so 32 uniform `500` m steps carry the column cleanly to `18,000` m AGL.

### Equation Bridge
$$d_0 = x_0, \qquad d_k = x_k - x_{k-1} \;\; (k = 1,\dots,43), \qquad x_k = \sum_{j=0}^{k} d_j$$
Notice that `from_diff(to_diff(prof))` is an exact telescoping sum: every intermediate term cancels, so the encoding is completely lossless up to floating-point rounding.

> **Predict → Run → Observe (Vertical-Difference Encoding)**
> - **Predict:** Before running the cell below, estimate: (1) how large the round-trip reconstruction error `max |prof - from_diff(to_diff(prof))|` will be in `float64`, and (2) when the jet stream reaches `~35 m/s` near 11.5 km, will the encoded step changes $\Delta u$ be closer to `~30 m/s` or `~2–4 m/s`?
> - **Run:** Execute the next cell to inspect `HEIGHTS`, `PROFILE`, and the three-panel wind comparison.
> - **What to observe:** Check the printed round-trip error (`~0` to `1e-14`, machine precision) and compare the horizontal axis scale of the **middle panel** (raw $u(z)$ spanning ~3–38 m/s) against the **right panel** (step changes $\Delta u$ tightly clustered around 0, flipping sign right above the jet core).

In [ ]:
# @title 1. Inspect the 44-Height Grid & Run to_diff / from_diff
import numpy as np
import matplotlib.pyplot as plt

from tempest.profiles.to_heights import HEIGHTS, PROFILE, geometric_height, liquid_rh
from tempest.profiles.diff import to_diff, from_diff
from tempest.profiles.thermo import column as thermo_column, es_liquid, gravity, virtual_t
from tempest_weather._core import site_shift

# Build a realistic synthetic Spaceport America summer sounding on the 44-height grid
z = HEIGHTS.astype(np.float64)  # metres AGL (44,)
u = 3.0 + 24.0 * np.exp(-0.5 * ((z - 11500.0) / 3200.0) ** 2) + 2.5 * np.log1p(z / 120.0)
v = -2.0 + 8.0 * np.sin(z / 4500.0)
T = np.where(z <= 12000.0, 303.0 - 0.0068 * z, 303.0 - 0.0068 * 12000.0 + 0.0010 * (z - 12000.0))
RH = np.clip(38.0 * np.exp(-z / 6500.0) + 18.0 * np.exp(-0.5 * ((z - 4500.0) / 1400.0) ** 2), 3.0, 98.0)

# Stack as (4, 44) matching PROFILE = ("u", "v", "T", "RH")
prof = np.stack([u, v, T, RH], axis=0)
encoded = to_diff(prof)
reconstructed = from_diff(encoded)
max_err = np.max(np.abs(prof - reconstructed))

print(f"HEIGHTS count: {len(HEIGHTS)} levels | min={HEIGHTS[0]:.0f} m, max={HEIGHTS[-1]:.0f} m AGL")
print(f"PROFILE order: {PROFILE}")
print(f"Round-trip max |prof - from_diff(to_diff(prof))|: {max_err:.2e}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].plot(np.arange(len(HEIGHTS)), HEIGHTS / 1000.0, "o-", ms=3, color="#2563eb")
axes[0].set(title="44-Level AGL Grid Spacing", xlabel="Level Index (0..43)", ylabel="Altitude (km AGL)")
axes[0].grid(alpha=0.3)

axes[1].plot(u, z / 1000.0, lw=2, color="#0ea5e9", label="Raw u(z)")
axes[1].plot(v, z / 1000.0, lw=2, color="#f97316", label="Raw v(z)")
axes[1].set(title="Physical Wind Profile", xlabel="Wind (m/s)", ylabel="Altitude (km AGL)")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].barh(np.arange(1, len(HEIGHTS)), encoded[0, 1:], color="#0ea5e9", alpha=0.8)
axes[2].axvline(0, color="k", lw=0.8)
axes[2].set(title="Encoded Vertical Steps Δu", xlabel="Step Δu (m/s per layer)", ylabel="Layer Index (1..43)")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Left:** Notice the dense cluster of 12 levels below 2 km AGL transitioning into uniform 500 m steps up to 18 km.
- **Middle vs. Right:** Raw eastward wind $u(z)$ spans 3 to 38 m/s across the jet stream, yet the layer-to-layer differences $\Delta u$ stay compact and sign-symmetric around zero—making vertical wind shear a first-class target for the neural network.

---

## 2. Log Dewpoint-Depression Humidity Coordinate (`dpd_from_rh` / `rh_from_dpd`)

### Conceptual Intuition (Before the Math)
Relative humidity ($\text{RH}$) lives inside a hard box between **0% and 100%**. Asking a generative neural network to predict $\text{RH}$ directly is like bowling without bumpers: stochastic noise pushes samples to `-12%` or `118%`, forcing hard clipping that piles artificial probability spikes onto 0% and 100%.

Tempest-ARKNET avoids that distortion by switching from *how full the air is* ($\text{RH}\%$) to *how many kelvin the air must cool before dew forms*—the **dewpoint depression** $T - T_d \ge 0$—compressed through `log1p`:
- **Why `log1p(T - Td)` instead of `log(T - Td)`?** At 100% saturation, $T - T_d = 0$; plain $\ln(0) = -\infty$ blows up gradients, whereas $\ln(1 + 0) = 0$ gives a finite, smooth saturation boundary while still compressing wide dry-air depressions (`T - Td = 40 K` maps to `lnd ≈ 3.7`).
- **Why the Magnus-Tetens constants (`611.21`, `17.502`, `273.16`, `32.19`)?** Those are the ECMWF liquid-water saturation vapor pressure constants ($e_{s,l}(273.16\text{ K}) = 611.21\text{ Pa}$), matching ERA5's `liquid_rh` convention so training targets and thermodynamic post-processing stay consistent.
- **Why floor `RH` at `0.5%` in `dpd_from_rh` and cap `lnd` at `5.0` in `rh_from_dpd`?** Flooring `RH >= 0.5%` prevents $\ln(0)$ overflow in bone-dry upper-tropospheric air. Capping `lnd <= 5.0` limits $T - T_d \le \mathrm{expm1}(5.0) \approx 147.4\text{ K}$, keeping $T_d$ safely away from the $32.19\text{ K}$ denominator pole of the Magnus formula during early noisy training.

### Equation Bridge (`tempest/generator/sample.py`)
$$e_{s,l}(T) = 611.21\exp\!\left(\frac{17.502\,(T - 273.16)}{T - 32.19}\right), \qquad \text{lnd} = \ln\!\big(1 + \max(T - T_d, 0)\big)$$
$$T_d = T - \max\!\big(\mathrm{expm1}(\min(\text{lnd}, 5.0)),\, 0\big), \qquad \mathrm{RH} = 100\,\frac{e_{s,l}(T_d)}{e_{s,l}(T)}$$
Because $\max(\cdot, 0)$ guarantees $T_d \le T$ and $e_{s,l}$ is strictly increasing in $T$, the recovered $\mathrm{RH}$ can **never** exceed 100%—even if the neural network outputs a negative `lnd`.

> **Predict → Run → Observe (Humidity Coordinate & Saturation Safety)**
> - **Predict:** Before running the next cell: (1) If a noisy neural network outputs `lnd = -1.0`, will `rh_from_dpd(300.0, -1.0)` return a supersaturated value (`> 100%`) or clamp cleanly to `100.00%`? (2) For the same `RH = 10%`, will warm air (`300 K`) or cold upper air (`240 K`) have a larger dewpoint depression `lnd`?
> - **Run:** Execute the next cell to evaluate both forward (`dpd_from_rh`) and inverse (`rh_from_dpd`) transforms.
> - **What to observe:** Verify the printed overshoot test at `lnd = -1.0`, inspect the flat plateau on the left side of the **right panel** (`lnd <= 0`), and notice how the `300 K` and `240 K` curves separate at low RH in the **left panel** because saturation vapor pressure scales exponentially with temperature.

In [ ]:
# @title 2. Compare Raw RH vs. Log Dewpoint-Depression Coordinate (lnd)
# On Google Colab (torch pre-installed), this imports directly from tempest.generator.sample.
try:
    from tempest.generator.sample import dpd_from_rh, rh_from_dpd, to_site
    SAMPLE_SOURCE = "tempest.generator.sample"
except ImportError:
    SAMPLE_SOURCE = "pure-NumPy equivalent (torch not installed)"
    def dpd_from_rh(t, rh):
        y = np.log(np.clip(rh, 0.5, 100.0) / 100.0 * es_liquid(t) / 611.21)
        td = (y * 32.19 - 17.502 * 273.16) / (y - 17.502)
        return np.log1p(np.maximum(t - td, 0.0))
    def rh_from_dpd(t, lnd):
        td = t - np.maximum(np.expm1(np.clip(lnd, None, 5.0)), 0.0)
        return 100.0 * es_liquid(td) / es_liquid(t)
    to_site = None

print(f"Using humidity coordinate functions from: {SAMPLE_SOURCE}")

rh_grid = np.linspace(1.0, 100.0, 200)
lnd_warm = dpd_from_rh(np.full_like(rh_grid, 300.0), rh_grid)
lnd_cold = dpd_from_rh(np.full_like(rh_grid, 240.0), rh_grid)

# Verify round-trip and negative-lnd saturation safety
rh_roundtrip = rh_from_dpd(300.0, lnd_warm)
lnd_test = np.linspace(-1.0, 4.5, 200)
rh_from_unconstrained = rh_from_dpd(300.0, lnd_test)

print(f"Max round-trip RH error (300 K): {np.max(np.abs(rh_grid - rh_roundtrip)):.2e} %")
print(f"Max RH when lnd = -1.0 (overshoot test): {rh_from_dpd(300.0, -1.0):.2f} % (guaranteed <= 100%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(rh_grid, lnd_warm, lw=2, color="#dc2626", label="T = 300 K (surface)")
axes[0].plot(rh_grid, lnd_cold, lw=2, color="#2563eb", label="T = 240 K (upper air)")
axes[0].set(title="Forward Transform: RH (%) → lnd", xlabel="Relative Humidity (%)", ylabel="lnd = ln(1 + T - Td)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(lnd_test, rh_from_unconstrained, lw=2.2, color="#16a34a")
axes[1].axvline(0.0, ls="--", color="gray", label="lnd = 0 (saturation boundary)")
axes[1].set(title="Inverse Transform: Unconstrained lnd → Bounded RH", xlabel="Model Output lnd", ylabel="Recovered RH (%)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Left:** Very dry air (1–10% RH) maps smoothly to `lnd ≈ 3.2–4.4` instead of crowding against zero, and warmer air requires a larger cooling ($T - T_d$) to reach saturation at the same RH.
- **Right:** Even when an unconstrained neural output overshoots to `lnd < 0`, `rh_from_dpd` clamps smoothly at 100% saturation with zero unphysical supersaturation.

---

## 3. 2-Pass Hypsometric Column & Interactive Site Elevation Shift

### Conceptual Intuition (Before the Math)
Rocket aerodynamic drag ($\frac{1}{2}\rho v^2 C_D A$) and Mach number ($v / a$) depend directly on **air density $\rho(z)$** and **speed of sound $a(z)$**, yet Tempest's neural generator never predicts pressure or density aloft. Why? Because pressure and density are *not independent degrees of freedom*—if a neural network predicted $(p, T, \rho)$ separately, small errors would violate hydrostatic balance and the ideal gas law.

Instead, `tempest/profiles/thermo.py::column` derives $(p, \rho, a)$ deterministically from $(T, \text{RH}, p_s)$:
- **Why Virtual Temperature $T_v$?** Water vapor ($\text{H}_2\text{O}$, molar mass ~18 g/mol) is lighter than dry air ($\text{N}_2/\text{O}_2$, ~29 g/mol). Replacing dry air molecules with water vapor makes the parcel *less dense* at the same $(T, p)$. Defining
  $$T_v = T\!\left(1 + \left(\frac{R_v}{R_d} - 1\right)q\right) \approx T(1 + 0.608\,q)$$
  with $R_d = 287.05\text{ J/(kg}\cdot\text{K)}$ and $R_v = 461.5\text{ J/(kg}\cdot\text{K)}$ lets us keep the dry-air gas constant $R_d$ in both density $\rho = p / (R_d T_v)$ and speed of sound $a = \sqrt{\gamma R_d T_v}$ ($\gamma = 1.4$).
- **Why the Hypsometric Equation & 2-Pass Fixed-Point Iteration?** Integrating hydrostatic balance $dp/dz = -\rho g$ across each vertical layer gives the exact hypsometric thickness relation:
  $$\ln p_{\text{top}} = \ln p_{\text{bot}} - \frac{g(z_{\text{mid}})\,\Delta z}{R_d\,\bar{T}_v}, \qquad g(z_{\text{ASL}}) = g_0\!\left(\frac{R_\oplus}{R_\oplus + z_{\text{ASL}}}\right)^2$$
  using $g_0 = 9.80665\text{ m/s}^2$ and $R_\oplus = 6{,}371{,}000\text{ m}$. Notice the chicken-and-egg loop: specific humidity $q(T, \text{RH}, p)$ needs $p$, while $p$ needs $T_v(q)$. **Pass 1** integrates dry pressure ($T_v = T$); **Pass 2** computes $q$ from Pass 1's pressure, updates $T_v$, and re-integrates—converging pressure to within $<0.01\%$.
- **Why Exponential Fade ($e^{-z/1500\text{ m}}$) in `to_site` / `site_shift`?** When your launch pad sits $\Delta z$ meters above the nearest ERA5 grid cell, surface pressure drops hypsometrically and 2 m temperature cools by the standard lapse rate $\Gamma = 0.0065\text{ K/m}$. Multiplying the aloft temperature shift by $w(z) = e^{-z / 1500\text{ m}}$ keeps the transition seamless from 2 m to 10 m while letting the terrain adjustment fade out in the free atmosphere above ~5 km. Updating $\text{RH}_{\text{new}} = \text{RH}_{\text{old}}\,e_s(T_{\text{old}})/e_s(T_{\text{new}})$ conserves the air's dew point (actual moisture content) as it cools.

> **Predict → Run → Observe (Elevation Shift & Hydrostatic Column)**
> - **Predict:** With the default sliders (`ELEV_SHIFT_M = +350 m`, moving the pad *up* from `1200 m` to `1550 m` MSL): (1) Will 10 m air density $\rho$ increase or decrease—and roughly by how many percent? (2) Because the shifted pad is cooler while conserving dew point, will surface Relative Humidity go **up** or **down**? (3) Above what altitude (in km AGL) will the dashed ERA5 grid curve and solid site curve visually merge in Temperature?
> - **Run:** Execute the next cell, then experiment with negative `ELEV_SHIFT_M` (e.g., `-400 m`) or higher `SURFACE_RH_PCT` (`85%`) to test your intuition.
> - **What to observe:** Check the printed percentage change in 10 m air density (about `-3%` to `-4%` for `+350 m`), confirm that RH rises near the cooler elevated pad, and verify that the Temperature shift fades cleanly to zero around `5–6 km AGL` while the density ratio persists slightly differently with altitude.

In [ ]:
# @title 3. Interactive Site-Elevation Shift & 2-Pass Hypsometric Column
ELEV_SHIFT_M = 350 # @param {type:"slider", min:-600, max:600, step:50}
SURFACE_TEMP_K = 302 # @param {type:"slider", min:265, max:315, step:1}
SURFACE_RH_PCT = 35 # @param {type:"slider", min:5, max:95, step:5}
GRID_ELEV_M = 1200 # @param {type:"slider", min:0, max:2500, step:100}

# Build a 179-element physical vector matching Tempest's canonical layout:
# [0:44]=u, [44:88]=v, [88:132]=T, [132:176]=RH, [176]=t2m, [177]=sp, [178]=rh2m
z_agl = HEIGHTS.astype(np.float64)
T_grid = np.where(z_agl <= 12000.0, SURFACE_TEMP_K - 0.0065 * z_agl, SURFACE_TEMP_K - 0.0065 * 12000.0 + 0.001 * (z_agl - 12000.0))
RH_grid = np.clip(SURFACE_RH_PCT * np.exp(-z_agl / 7000.0) + 10.0, 2.0, 99.0)
sp_grid = 101325.0 * (1.0 - 2.25577e-5 * GRID_ELEV_M) ** 5.25588

phys_grid = np.concatenate([u, v, T_grid, RH_grid, [SURFACE_TEMP_K, sp_grid, SURFACE_RH_PCT]]).astype(np.float64)
phys_site = phys_grid.copy()

z_site = GRID_ELEV_M + ELEV_SHIFT_M
if to_site is not None:
    to_site(phys_site, GRID_ELEV_M, z_site)
else:
    # Pure-NumPy equivalent from tempest_weather._core.site_shift
    phys_site = site_shift(phys_site[None, :], float(ELEV_SHIFT_M))[0]

# Run 2-pass hypsometric thermo.column on both grid and shifted site profiles
col_grid = thermo_column(
    z_agl, phys_grid[88:132], phys_grid[132:176], phys_grid[177],
    t_sfc=phys_grid[176], rh_sfc=phys_grid[178], z_sfc=GRID_ELEV_M
)
col_site = thermo_column(
    z_agl, phys_site[88:132], phys_site[132:176], phys_site[177],
    t_sfc=phys_site[176], rh_sfc=phys_site[178], z_sfc=z_site
)
p_grid, rho_grid, a_grid = col_grid["p"], col_grid["rho"], col_grid["a"]
p_site, rho_site, a_site = col_site["p"], col_site["rho"], col_site["a"]

print(f"Grid elevation: {GRID_ELEV_M} m MSL -> Site elevation: {z_site} m MSL (Δz = {ELEV_SHIFT_M:+d} m)")
print(f"Surface pressure: {phys_grid[177]/100:.1f} hPa -> {phys_site[177]/100:.1f} hPa")
print(f"10 m air density: {rho_grid[0]:.4f} kg/m³ -> {rho_site[0]:.4f} kg/m³ ({100*(rho_site[0]/rho_grid[0]-1):+.2f}%)")
print(f"10 m speed of sound: {a_grid[0]:.2f} m/s -> {a_site[0]:.2f} m/s")

fig, axes = plt.subplots(1, 4, figsize=(16, 4.6), sharey=True)
panels = [
    ("Temperature (K)", phys_grid[88:132], phys_site[88:132], "#dc2626"),
    ("Relative Humidity (%)", phys_grid[132:176], phys_site[132:176], "#0284c7"),
    ("Air Density (kg/m³)", rho_grid, rho_site, "#7c3aed"),
    ("Speed of Sound (m/s)", a_grid, a_site, "#059669"),
]
for ax, (title, g_val, s_val, col) in zip(axes, panels):
    ax.plot(g_val, z_agl / 1000.0, "--", lw=1.8, color="gray", label="ERA5 Grid")
    ax.plot(s_val, z_agl / 1000.0, lw=2.2, color=col, label=f"Site ({ELEV_SHIFT_M:+d} m)")
    ax.set(title=title, xlabel=title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Altitude (km AGL)")
plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Temperature & Humidity:** The temperature shift is strongest right at the launch pad (`0–2 km AGL`) and fades exponentially ($e^{-z/1500\text{ m}}$) so the upper atmosphere above ~5 km stays untouched, while RH rises when moving to a cooler, higher pad (conserving dew point).
- **Air Density & Speed of Sound:** Moving the pad higher by `+350 m` lowers surface air density by ~3–4% and slightly lowers speed of sound (via cooler $T_v$), which directly reduces aerodynamic drag and shifts the rocket's transonic Mach transition during ascent.